In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from pgmpy.estimators import BIC, BayesianEstimator, HillClimbSearch
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")


# Load data in a DataFrame
data = pd.read_excel("./Dataset1_BankClients.xlsx")
# Drop the column by its actual name (e.g., 'ID' or the actual name of the column)
data = data.drop(columns=["ID"])  # Replace 'ID' with the actual column name to drop

# Define Business Mapping dictionary
mapping_dicts = {
    "Gender": {"0": "Male", "1": "Female", "0.0": "Male", "1.0": "Female"},
    "Job": {
        "1": "Unemployed",
        "2": "Employee/Worker",
        "3": "Manager/Executive",
        "4": "Entrepreneur/Freelancer",
        "5": "Retired",
    },
    "Area": {"1": "Nord", "2": "Centro", "3": "Sud/Isole"},
    "CitySize": {"1": "Small town", "2": "Medium-sized city", "3": "Large city"},
    "Investments": {
        "1": "No investments",
        "2": "Mostly lump sum",
        "3": "Capital accumulation",
    },
}

## Data processing


### One hot encoding


In [ ]:
# Specify categorical variables
categorical_columns = ["Gender", "Job", "Area", "CitySize", "Investments"]

# Apply one-hot encoding to categorical features
data_encoded = pd.get_dummies(data, columns=categorical_columns, dtype=int)

### Scaling


In [ ]:
# Identify continuous numeric columns from the original data (exclude categorical columns).
continuous_columns = [
    col
    for col in data.select_dtypes(include=["number"]).columns
    if col not in categorical_columns
]

# Scale only continuous columns and keep one-hot columns unchanged.
scaler = StandardScaler()
data_scaled = data_encoded.copy()
data_scaled[continuous_columns] = scaler.fit_transform(data_scaled[continuous_columns])

data_scaled.head()

## Clustering


### Adaptive Clustering with Bayesian Gaussian Mixture (BGM)

- Set an upper bound of `K = 15` components.
- Let the Dirichlet Process prior auto-prune unsupported clusters.
- Keep only clusters with meaningful weights (default threshold: `> 5%`) as final Persona frameworks.


In [ ]:
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture

# Set upper limit of candidate clusters
max_clusters = 15

# Train BGM with Dirichlet Process prior for auto-pruning
bgm = BayesianGaussianMixture(
    n_components=max_clusters,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    random_state=42,
    max_iter=1000,
    init_params="kmeans",
)

bgm.fit(data_scaled)

In [ ]:
# Raw cluster assignment from all components
raw_labels = bgm.predict(data_scaled)
raw_weights = pd.Series(bgm.weights_, name="weight")

# Lock final personas by removing low-weight components
weight_threshold = 0.05  # only keep components with > 5% weight
active_components = raw_weights[raw_weights > weight_threshold].index.tolist()

# Keep only samples assigned to active components
active_mask = pd.Series(raw_labels).isin(active_components)
data_persona = data_scaled.loc[active_mask].copy()
data_persona["persona_cluster"] = pd.Series(raw_labels)[active_mask].values

# Re-map cluster ids to consecutive labels (0..n-1) for readability
cluster_remap = {old: new for new, old in enumerate(sorted(active_components))}
data_persona["persona_cluster"] = data_persona["persona_cluster"].map(cluster_remap)

# Summary table of retained persona frameworks
persona_summary = (
    data_persona["persona_cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
    .to_frame()
)
persona_summary["count"] = (
    data_persona["persona_cluster"].value_counts().sort_index().values
)

print(f"Max components (K): {max_clusters}")
print(f"Retained persona frameworks: {len(active_components)}")
print(f"Active original component IDs: {sorted(active_components)}")

persona_summary

## Basic Profiling & Naming


### Qualitative Overlay: Rebuild Centroid Features into Business Tags

- Recover continuous centroid means in original units.
- Decode dominant one-hot categories for each retained cluster.
- Produce business-readable profiles to support persona naming.


In [ ]:
# Cluster means on scaled data
cluster_scaled_means = data_persona.groupby("persona_cluster").mean(numeric_only=True)

# Restore continuous means to original units
continuous_scaled = cluster_scaled_means[continuous_columns]
continuous_original = pd.DataFrame(
    scaler.inverse_transform(continuous_scaled),
    columns=continuous_columns,
    index=continuous_scaled.index,
)

# Decode dominant category from one-hot proportions
categorical_profile = pd.DataFrame(index=cluster_scaled_means.index)
for col in categorical_columns:
    onehot_cols = [c for c in cluster_scaled_means.columns if c.startswith(f"{col}_")]
    if not onehot_cols:
        continue

    proportions = cluster_scaled_means[onehot_cols]
    dominant_onehot = proportions.idxmax(axis=1)
    dominant_value_raw = dominant_onehot.str.replace(f"{col}_", "", regex=False)
    dominant_value_mapped = dominant_value_raw.map(
        lambda x: mapping_dicts.get(col, {}).get(str(x), x)
    )
    dominant_share = proportions.max(axis=1)

    categorical_profile[f"{col}_dominant"] = dominant_value_mapped
    categorical_profile[f"{col}_share"] = dominant_share

# Build base business profile and add cluster coverage metrics
business_profile = continuous_original.join(categorical_profile)
cluster_count = data_persona["persona_cluster"].value_counts().sort_index()
cluster_share = cluster_count / cluster_count.sum()
business_profile["persona_size"] = (
    cluster_count.reindex(business_profile.index).fillna(0).astype(int)
)
business_profile["persona_share"] = cluster_share.reindex(
    business_profile.index
).fillna(0.0)


# Categorical confidence helps flag whether dominant categories are clear or mixed.
def share_confidence_tag(v: float) -> str:
    if v >= 0.75:
        return "High confidence"
    if v >= 0.55:
        return "Medium confidence"
    return "Low confidence"


for col in categorical_columns:
    share_col = f"{col}_share"
    if share_col in business_profile.columns:
        business_profile[f"{col}_confidence"] = business_profile[share_col].apply(
            share_confidence_tag
        )

business_profile

In [ ]:
def age_band(age_value: float) -> str:
    if age_value < 35:
        return "18-34"
    if age_value < 55:
        return "35-54"
    if age_value <= 70:
        return "55-70"
    return "70+"


def pct_to_tag(val: float, high_thresh=0.66, low_thresh=0.33) -> str:
    if val >= high_thresh:
        return "High"
    if val <= low_thresh:
        return "Low"
    return "Medium"

In [ ]:
def safe_text_series(df: pd.DataFrame, col: str, default: str = "Unknown") -> pd.Series:
    if col in df.columns:
        return df[col].astype(str)
    return pd.Series(default, index=df.index, dtype="object")


business_profile["Age_band"] = business_profile["Age"].apply(age_band)

# Use cluster-relative quantiles for richer segmentation tags.
feature_thresholds = {}
for feat in [
    "Wealth",
    "Debt",
    "Digital",
    "Income",
    "FinEdu",
    "ESG",
    "BankFriend",
    "LifeStyle",
    "Luxury",
    "Saving",
]:
    if feat in business_profile.columns:
        q_low, q_high = business_profile[feat].quantile([0.33, 0.66]).values
        feature_thresholds[feat] = (float(q_low), float(q_high))

if "Wealth" in business_profile.columns:
    q_low, q_high = feature_thresholds["Wealth"]
    business_profile["Wealth_tag"] = business_profile["Wealth"].apply(
        lambda x: pct_to_tag(x, q_high, q_low) + " Wealth"
    )
if "Debt" in business_profile.columns:
    q_low, q_high = feature_thresholds["Debt"]
    business_profile["Debt_tag"] = business_profile["Debt"].apply(
        lambda x: pct_to_tag(x, q_high, q_low) + " Debt"
    )
if "Digital" in business_profile.columns:
    q_low, q_high = feature_thresholds["Digital"]
    business_profile["Digital_tag"] = business_profile["Digital"].apply(
        lambda x: pct_to_tag(x, q_high, q_low) + " Digital"
    )

extra_percentile_features = [
    "Income",
    "FinEdu",
    "ESG",
    "BankFriend",
    "LifeStyle",
    "Luxury",
    "Saving",
]
for feat in extra_percentile_features:
    if feat in business_profile.columns:
        q_low, q_high = feature_thresholds[feat]
        business_profile[f"{feat}_tag"] = business_profile[feat].apply(
            lambda x: pct_to_tag(x, q_high, q_low) + f" {feat}"
        )

# Composite indicators for richer profile interpretation.
score_map = {"Low": 0, "Medium": 1, "High": 2}
for feat in ["Wealth", "Saving", "Debt", "Digital", "FinEdu", "BankFriend"]:
    tag_col = f"{feat}_tag"
    if tag_col in business_profile.columns:
        business_profile[f"{feat}_score"] = (
            business_profile[tag_col].str.split().str[0].map(score_map).fillna(1)
        )

if {"Wealth_score", "Saving_score", "Debt_score"}.issubset(business_profile.columns):
    resilience_raw = (
        business_profile["Wealth_score"]
        + business_profile["Saving_score"]
        + (2 - business_profile["Debt_score"])
    )
    business_profile["Financial_resilience"] = pd.cut(
        resilience_raw,
        bins=[-np.inf, 2.5, 4.5, np.inf],
        labels=["Fragile", "Balanced", "Strong"],
    ).astype(str)

if {"Digital_score", "FinEdu_score", "BankFriend_score"}.issubset(
    business_profile.columns
):
    engagement_raw = (
        business_profile["Digital_score"]
        + business_profile["FinEdu_score"]
        + business_profile["BankFriend_score"]
    )
    business_profile["Engagement_style"] = pd.cut(
        engagement_raw,
        bins=[-np.inf, 2.5, 4.5, np.inf],
        labels=["Low-touch", "Hybrid", "Advisor-led"],
    ).astype(str)

# Build qualitative overlay with safe fallback columns.
business_profile["Qualitative_overlay"] = (
    "Age "
    + safe_text_series(business_profile, "Age_band")
    + " | "
    + safe_text_series(business_profile, "Gender_dominant")
    + " | "
    + safe_text_series(business_profile, "Job_dominant")
    + " | "
    + safe_text_series(business_profile, "Wealth_tag")
    + " | Debt: "
    + safe_text_series(business_profile, "Debt_tag").str.replace(
        " Debt", "", regex=False
    )
    + " | Digital: "
    + safe_text_series(business_profile, "Digital_tag").str.replace(
        " Digital", "", regex=False
    )
    + " | Resilience: "
    + safe_text_series(business_profile, "Financial_resilience")
    + " | Service model: "
    + safe_text_series(business_profile, "Engagement_style")
)

business_profile

In [ ]:
# Optional: manual business naming column to fill in after review
business_profile["Suggested_name"] = business_profile.get("Suggested_name", "")

# Display high-value columns first
display_cols = [
    "persona_size",
    "persona_share",
    "Age",
    "Age_band",
    "Gender_dominant",
    "Gender_confidence",
    "Job_dominant",
    "Job_confidence",
    "Investments_dominant",
    "Investments_confidence",
    "Area_dominant",
    "CitySize_dominant",
    # Financial and liability status
    "Wealth_tag",
    "Income_tag",
    "Debt_tag",
    "Saving_tag",
    "Financial_resilience",
    # Cognitive and digital preferences
    "Digital_tag",
    "FinEdu_tag",
    "Engagement_style",
    # Consumption and values
    "Luxury_tag",
    "LifeStyle_tag",
    "ESG_tag",
    # Social and relational context
    "BankFriend_tag",
    "Qualitative_overlay",
    "Suggested_name",
]

existing_cols = [c for c in display_cols if c in business_profile.columns]
persona_profile_view = business_profile[existing_cols].copy()
persona_profile_view["persona_share_pct"] = (
    persona_profile_view["persona_share"].fillna(0).mul(100).round(2).astype(str) + "%"
)

persona_profile_view

In [ ]:
business_profile["Qualitative_overlay"]

## Introducing Bayesian Networks for Deep Insight and Inference


In [ ]:
# 1) Select a target persona subgroup with weighted business relevance
profile = business_profile.copy()
profile["_age_score"] = (
    profile["Age"].rank(pct=True) if "Age" in profile.columns else 0.0
)
profile["_debt_score"] = (
    (-profile["Debt"]).rank(pct=True) if "Debt" in profile.columns else 0.0
)
profile["_wealth_score"] = (
    profile["Wealth"].rank(pct=True) if "Wealth" in profile.columns else 0.0
)

female_aliases = {"female", "f", "1", "1.0"}
if "Gender_dominant" in profile.columns:
    profile["_gender_score"] = (
        profile["Gender_dominant"]
        .astype(str)
        .str.lower()
        .isin(female_aliases)
        .astype(float)
    )
else:
    profile["_gender_score"] = 0.0

profile["_persona_score"] = (
    0.35 * profile["_age_score"]
    + 0.30 * profile["_debt_score"]
    + 0.25 * profile["_wealth_score"]
    + 0.10 * profile["_gender_score"]
)

focus_cluster = int(profile["_persona_score"].idxmax())
mask_focus = data_persona["persona_cluster"] == focus_cluster
focus_idx = data_persona[mask_focus].index
focus_raw = data.loc[focus_idx].copy()

# 2) Build a discrete dataset for BN structure learning
bn_data = pd.DataFrame(index=focus_raw.index)

if "Age" in focus_raw.columns:
    bn_data["Age_band"] = pd.cut(
        focus_raw["Age"],
        bins=[-np.inf, 34, 54, 70, np.inf],
        labels=["18-34", "35-54", "55-70", "70+"],
        include_lowest=True,
    ).astype(str)

candidate_cats = [
    "Gender",
    "Job",
    "Area",
    "CitySize",
    "Investments",
    "MaritalStatus",
    "Mortgage",
]
for c in candidate_cats:
    if c in focus_raw.columns:
        bn_data[c] = focus_raw[c].astype(str)

# Discretize key continuous features into robust tertiles for BN compatibility.
numeric_to_discretize = [
    "Wealth",
    "Income",
    "Debt",
    "Digital",
    "Saving",
    "FinEdu",
    "ESG",
    "BankFriend",
    "LifeStyle",
    "Luxury",
]
for c in numeric_to_discretize:
    if c in focus_raw.columns:
        q_low, q_high = focus_raw[c].quantile([0.33, 0.66]).values
        if q_low == q_high:
            bn_data[f"{c}_level"] = pd.cut(
                focus_raw[c],
                bins=3,
                labels=["Low", "Medium", "High"],
                include_lowest=True,
            ).astype(str)
        else:
            bn_data[f"{c}_level"] = pd.cut(
                focus_raw[c],
                bins=[-np.inf, q_low, q_high, np.inf],
                labels=["Low", "Medium", "High"],
                include_lowest=True,
            ).astype(str)

# Need target for inference
need_derived_from_investments = False
if {"LowRiskInvestments", "HighRiskStocks"}.issubset(set(focus_raw.columns)):
    bn_data["Need"] = np.where(
        focus_raw["LowRiskInvestments"] >= focus_raw["HighRiskStocks"],
        "LowRiskInvestments",
        "HighRiskStocks",
    )
elif "Investments" in focus_raw.columns:
    inv_vals = focus_raw["Investments"]
    bn_data["Need"] = np.select(
        [inv_vals == inv_vals.min(), inv_vals == inv_vals.max()],
        ["LowRiskInvestments", "HighRiskStocks"],
        default="Balanced",
    )
    need_derived_from_investments = True
else:
    raise ValueError(
        "Need proxy missing. Add Investments or explicit risk-preference columns."
    )

# Avoid target leakage
if need_derived_from_investments and "Investments" in bn_data.columns:
    bn_data = bn_data.drop(columns=["Investments"])

bn_data = bn_data.dropna().copy()
if bn_data.empty:
    raise ValueError("No records available for BN after preprocessing.")

# Merge very rare states to increase BN stability
for col in bn_data.columns:
    vc = bn_data[col].value_counts(normalize=True)
    rare = vc[vc < 0.02].index
    if len(rare) > 0:
        bn_data[col] = bn_data[col].replace(rare, "Other")

for col in bn_data.columns:
    bn_data[col] = pd.Categorical(bn_data[col].astype(str))

if "Need" not in bn_data.columns:
    raise ValueError("Need variable is required for inference.")

# 3) Structure learning
hc = HillClimbSearch(bn_data)
learned_structure = hc.estimate(scoring_method=BIC(bn_data), max_indegree=3)
learned_edges = list(learned_structure.edges())

# If Need is isolated, connect top informative predictors.
need_incident = [e for e in learned_edges if "Need" in e]
if len(need_incident) == 0:
    predictors = [c for c in bn_data.columns if c != "Need"]
    if len(predictors) > 0:
        X = pd.DataFrame({c: pd.factorize(bn_data[c])[0] for c in predictors})
        y = pd.factorize(bn_data["Need"])[0]
        mi = mutual_info_classif(X, y, discrete_features=True, random_state=42)
        mi_series = pd.Series(mi, index=predictors).sort_values(ascending=False)
        top_predictors = mi_series[mi_series > 1e-8].head(3).index.tolist()
        if len(top_predictors) == 0:
            top_predictors = predictors[: min(2, len(predictors))]
        learned_edges.extend([(p, "Need") for p in top_predictors])

model_nodes = sorted(
    set(["Need"] + [u for u, _ in learned_edges] + [v for _, v in learned_edges])
)
bn_train = bn_data[model_nodes].copy()

bn_model = DiscreteBayesianNetwork(learned_edges)
if "Need" not in bn_model.nodes():
    bn_model.add_node("Need")

bn_model.fit(
    bn_train,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10,
)

# 4) Visualize learned DAG
plt.figure(figsize=(11, 7))
G = nx.DiGraph()
G.add_nodes_from(bn_model.nodes())
G.add_edges_from(bn_model.edges())
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2500,
    node_color="#f5d76e",
    font_size=9,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=14,
)
plt.title(f"Learned Bayesian Network DAG | Persona Cluster {focus_cluster}")
plt.axis("off")
plt.show()

# 5) Probabilistic inference + scenario comparison
infer = VariableElimination(bn_model)
need_query = infer.query(variables=["Need"], show_progress=False)
need_prob = pd.Series(
    need_query.values, index=need_query.state_names["Need"], name="P(Need)"
).sort_values(ascending=False)

# Parent importance into Need via MI proxy (if available in training data)
bn_edge_report = pd.DataFrame(list(bn_model.edges()), columns=["from", "to"])
need_parents = [u for u, v in bn_model.edges() if v == "Need"]
if need_parents:
    mi_vals = []
    y_need = pd.factorize(bn_train["Need"])[0]
    for p in need_parents:
        x_parent = pd.factorize(bn_train[p])[0]
        mi_val = mutual_info_classif(
            pd.DataFrame({p: x_parent}), y_need, discrete_features=True, random_state=42
        )[0]
        mi_vals.append({"parent": p, "mutual_information": float(mi_val)})
    bn_edge_report = pd.DataFrame(mi_vals).sort_values(
        "mutual_information", ascending=False
    )

scenario_candidates = {
    "Baseline": {},
    "Conservative": {
        "Age_band": "55-70",
        "Debt_level": "Low",
        "Digital_level": "Low",
        "Wealth_level": "High",
    },
    "Growth_oriented": {
        "Age_band": "35-54",
        "Debt_level": "Medium",
        "Digital_level": "High",
        "Wealth_level": "High",
    },
}

scenario_results = []
for scenario_name, candidate_evidence in scenario_candidates.items():
    evidence = {}
    for focus_col, preferred in candidate_evidence.items():
        if focus_col in bn_model.nodes():
            cpd = bn_model.get_cpds(focus_col)
            if cpd is not None:
                states = list(cpd.state_names[focus_col])
                evidence[focus_col] = preferred if preferred in states else states[0]

    query = infer.query(variables=["Need"], evidence=evidence, show_progress=False)
    probs = pd.Series(query.values, index=query.state_names["Need"]).sort_values(
        ascending=False
    )
    top_need = str(probs.index[0])
    top_prob = float(probs.iloc[0])

    scenario_results.append(
        {
            "scenario": scenario_name,
            "applied_evidence": evidence,
            "top_need": top_need,
            "top_need_prob": top_prob,
        }
    )

bn_scenario_report = pd.DataFrame(scenario_results).sort_values(
    "top_need_prob", ascending=False
)
best_scenario = bn_scenario_report.iloc[0].to_dict()

print(f"Selected persona cluster for BN analysis: {focus_cluster}")
print(f"Subgroup size used for BN: {len(bn_train)}")
print("Need derived from Investments:", need_derived_from_investments)
print("Top unconditional Need:", need_prob.index[0], f"({need_prob.iloc[0]:.2%})")
print("Need parents:", need_parents)

need_prob.to_frame()

In [ ]:
import os

# 6) Final consolidated profile report
profile_report = persona_profile_view.copy()

if "persona_share" in profile_report.columns:
    profile_report["persona_share_pct"] = (
        profile_report["persona_share"].fillna(0).mul(100).round(2)
    )

profile_report["BN_focus_cluster"] = "No"
profile_report["BN_top_need"] = ""
profile_report["BN_best_scenario"] = ""

if "focus_cluster" in globals() and focus_cluster in profile_report.index:
    profile_report.loc[focus_cluster, "BN_focus_cluster"] = "Yes"

if "need_prob" in globals() and len(need_prob) > 0 and "focus_cluster" in globals():
    top_need = str(need_prob.index[0])
    top_prob = float(need_prob.iloc[0])
    if focus_cluster in profile_report.index:
        profile_report.loc[focus_cluster, "BN_top_need"] = (
            f"{top_need} ({top_prob:.2%})"
        )

if "best_scenario" in globals() and "focus_cluster" in globals():
    scenario_name = str(best_scenario.get("scenario", ""))
    if focus_cluster in profile_report.index:
        profile_report.loc[focus_cluster, "BN_best_scenario"] = scenario_name

# Persona-level narrative for business communication
narrative_lines = []
for cluster_id, row in profile_report.sort_index().iterrows():
    share_pct = row.get("persona_share_pct", np.nan)
    share_txt = f"{share_pct:.2f}%" if pd.notna(share_pct) else "n/a"

    narrative = (
        f"Persona {cluster_id}: {share_txt} of client base, "
        f"{row.get('Age_band', 'Unknown')} | {row.get('Job_dominant', 'Unknown')} | "
        f"{row.get('Financial_resilience', 'Unknown')} resilience | "
        f"{row.get('Engagement_style', 'Unknown')} engagement."
    )

    if row.get("BN_focus_cluster", "No") == "Yes":
        narrative += (
            f" BN suggests top product need: {row.get('BN_top_need', '')}, "
            f"best scenario: {row.get('BN_best_scenario', '')}."
        )

    narrative_lines.append({"persona_cluster": cluster_id, "narrative": narrative})

persona_narrative_report = pd.DataFrame(narrative_lines).set_index("persona_cluster")

# Save report artifacts
output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)
profile_report.to_csv(f"{output_dir}/persona_profile_report.csv")
persona_narrative_report.to_csv(f"{output_dir}/persona_profile_narrative.csv")

print("Saved:", f"{output_dir}/persona_profile_report.csv")
print("Saved:", f"{output_dir}/persona_profile_narrative.csv")

profile_report.sort_index()